In [10]:
import re

# ===========================================================================
# MISSING DATA STRUCTURES & REPRESENTATIONS
# ===========================================================================

class Clause:
    """Represents a definite clause rule: Premise1 & Premise2 ... => Conclusion"""
    def __init__(self, premises, conclusion):
        self.premises = premises      # List of tuple predicates
        self.conclusion = conclusion  # Tuple predicate

def is_variable(x):
    """By convention in FOL, variables are lowercase strings (e.g., 'x', 'y')"""
    return isinstance(x, str) and x.islower()

def substitute(theta, predicate):
    """Applies a substitution dictionary theta to a predicate tuple"""
    pred_name = predicate[0]
    args = [theta.get(arg, arg) if is_variable(arg) else arg for arg in predicate[1:]]
    return (pred_name,) + tuple(args)

def unify(p, q, theta=None):
    """Simplistic predicate unification matching predicate name and binding variables"""
    if theta is None:
        theta = {}
    if len(p) != len(q) or p[0] != q[0]:
        return None

    new_theta = theta.copy()
    for arg_p, arg_q in zip(p[1:], q[1:]):
        val_p = new_theta.get(arg_p, arg_p) if is_variable(arg_p) else arg_p
        val_q = new_theta.get(arg_q, arg_q) if is_variable(arg_q) else arg_q

        if val_p != val_q:
            if is_variable(val_p):
                new_theta[val_p] = val_q
            elif is_variable(val_q):
                new_theta[val_q] = val_p
            else:
                return None  # Constant mismatch
    return new_theta

# ===========================================================================
# INFERENCE ENGINE
# ===========================================================================

def forward_chain(rules, facts, goal):
    """
    Implements the core FOL-FC-ASK engine.
    Iteratively evaluates rules using unification matching to infer new ground facts.
    """
    current_facts = set(facts)
    print(f"Initial ground facts inside Knowledge Base:")
    for f in current_facts:
        print(f" - {f[0]}({', '.join(f[1:])})")

    iteration = 1
    while True:
        new_facts = set()
        rules_triggered_this_round = False

        for rule in rules:
            # Generate all possible variable bindings that satisfy the rule premises
            bindings_list = [{}]

            for premise in rule.premises:
                next_bindings = []
                for theta in bindings_list:
                    # Apply historical substitutions to the premise
                    subst_premise = substitute(theta, premise)

                    # Unify the structural rule premise with every known ground fact
                    for fact in current_facts:
                        unified_theta = unify(subst_premise, fact, theta)
                        if unified_theta is not None:
                            next_bindings.append(unified_theta)
                bindings_list = next_bindings

            # Fire rules for valid, unified variable structures
            for theta in bindings_list:
                inferred = substitute(theta, rule.conclusion)

                # Check for open variables remaining unassigned
                if any(is_variable(arg) for arg in inferred[1:]):
                    continue

                if inferred not in current_facts and inferred not in new_facts:
                    print(f"\n[Iteration {iteration}] Rule Fired: Using bindings {theta}")
                    print(f"   Inferred: {inferred[0]}({', '.join(inferred[1:])})")
                    new_facts.add(inferred)
                    rules_triggered_this_round = True

                    # Target Unification verification check
                    if unify(inferred, goal) is not None:
                        print(f"\n🎉 GOAL SUCCESS: Successfully proved {goal[0]}({', '.join(goal[1:])})!")
                        return True

        if not rules_triggered_this_round:
            break

        current_facts.update(new_facts)
        iteration += 1

    print(f"\n❌ GOAL FAILURE: Could not prove {goal[0]}({', '.join(goal[1:])}).")
    return False

# ===========================================================================
# PARSING METHODS
# ===========================================================================

def parse_predicate(pred_str):
    """Parses 'Predicate(Arg1, Arg2)' into ('Predicate', 'Arg1', 'Arg2')"""
    match = re.match(r"(\w+)\s*\(([^)]+)\)", pred_str.strip())
    if not match:
        raise ValueError(f"Invalid predicate format: '{pred_str}'")
    pred_name = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (pred_name,) + tuple(args)

def parse_rule(rule_str):
    """Parses 'P1(x) & P2(y) => C(x)' into a Clause object"""
    if "=>" not in rule_str:
        raise ValueError(f"Invalid rule format (missing '=>' operator): '{rule_str}'")
    premises_part, conclusion_part = rule_str.split("=>")

    premises_list = [p.strip() for p in premises_part.split("&") if p.strip()]
    premises = [parse_predicate(p) for p in premises_list]
    conclusion = parse_predicate(conclusion_part.strip())

    return Clause(premises, conclusion)

# ===========================================================================
# MAIN INTERACTIVE ENTRY POINT
# ===========================================================================


# Input facts
def_facts = "American(Robert), Enemy(CountryA, America), Owns(CountryA, Missiles), Missile(Missiles)"
facts_input = input(f"Enter facts separated by commas\n[Default: {def_facts}]: ").strip()
if not facts_input:
    facts_input = def_facts

parsed_facts = []
for f in facts_input.split("),"):
    f_str = f.strip()
    if f_str:
        if not f_str.endswith(")"):
            f_str += ")"
        parsed_facts.append(parse_predicate(f_str))

# Input rules
def_rules = [
    "American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z) => Criminal(x)",
    "Enemy(z, America) => Hostile(z)",
    "Missile(x) => Weapon(x)",
    "Missile(y) & Owns(z, y) => Sells(Robert, y, z)"
]

parsed_rules = []
print("\nEnter rules one by one. Press Enter on an empty line when finished.")
rule_index = 1
while True:
    default_val = def_rules[rule_index - 1] if rule_index <= len(def_rules) else ""
    prompt_suffix = f" [Default: {default_val}]" if default_val else ""

    r_in = input(f"Rule {rule_index}{prompt_suffix}: ").strip()
    if not r_in:
        if default_val:
            r_in = default_val
        else:
            break
    try:
        parsed_rules.append(parse_rule(r_in))
        rule_index += 1
    except ValueError as e:
        print(f"Error: {e}. Please re-enter this rule using valid syntax or press enter on an empty line to finish.")

# Input Goal
def_goal = "Criminal(Robert)"
goal_input = input(f"\nEnter the goal to prove\n[Default: {def_goal}]: ").strip()
if not goal_input:
    goal_input = def_goal
parsed_goal = parse_predicate(goal_input)

# Execute Engine
print("\n================ RUNNING FORWARD CHAINING ================")
forward_chain(parsed_rules, parsed_facts, parsed_goal)


Enter facts separated by commas
[Default: American(Robert), Enemy(CountryA, America), Owns(CountryA, Missiles), Missile(Missiles)]: 

Enter rules one by one. Press Enter on an empty line when finished.
Rule 1 [Default: American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z) => Criminal(x)]: 
Rule 2 [Default: Enemy(z, America) => Hostile(z)]: 
Rule 3 [Default: Missile(x) => Weapon(x)]: 
Rule 4 [Default: Missile(y) & Owns(z, y) => Sells(Robert, y, z)]: 
Rule 5: 

Enter the goal to prove
[Default: Criminal(Robert)]: 

================ RUNNING FORWARD CHAINING ================
Initial ground facts inside Knowledge Base:
 - American(Robert)
 - Owns(CountryA, Missiles)
 - Missile(Missiles)
 - Enemy(CountryA, America)

[Iteration 1] Rule Fired: Using bindings {'z': 'CountryA'}
   Inferred: Hostile(CountryA)

[Iteration 1] Rule Fired: Using bindings {'x': 'Missiles'}
   Inferred: Weapon(Missiles)

[Iteration 1] Rule Fired: Using bindings {'y': 'Missiles', 'z': 'CountryA'}
   Inferred: Sells(Robe

True

In [9]:
import re

# ===========================================================================
# DATA STRUCTURES & LOGIC REPRESENTATIONS
# ===========================================================================

class Clause:
    """Represents a definite clause rule: Premise1 & Premise2 ... => Conclusion"""
    def __init__(self, premises, conclusion):
        self.premises = premises      # List of tuple predicates
        self.conclusion = conclusion  # Tuple predicate

def is_variable(x):
    """By convention in FOL, variables are lowercase strings (e.g., 'x', 'y')"""
    return isinstance(x, str) and x.islower()

def substitute(theta, predicate):
    """Applies a substitution dictionary theta to a predicate tuple"""
    pred_name = predicate[0]
    args = [theta.get(arg, arg) if is_variable(arg) else arg for arg in predicate[1:]]
    return (pred_name,) + tuple(args)

def unify(p, q, theta=None):
    """Substance unification matching predicate name and binding variables"""
    if theta is None:
        theta = {}
    if p[0] != q[0] or len(p) != len(q):
        return None

    new_theta = theta.copy()
    for arg_p, arg_q in zip(p[1:], q[1:]):
        val_p = new_theta.get(arg_p, arg_p) if is_variable(arg_p) else arg_p
        val_q = new_theta.get(arg_q, arg_q) if is_variable(arg_q) else arg_q

        if val_p != val_q:
            if is_variable(val_p):
                new_theta[val_p] = val_q
            elif is_variable(val_q):
                new_theta[val_q] = val_p
            else:
                return None  # Constant mismatch
    return new_theta

# ===========================================================================
# BACKWARD CHAINING INFERENCE ENGINE
# ===========================================================================

def fol_bc_ask(rules, facts, goals, theta, depth=0):
    """
    Implements a recursive Depth-First Search Backward Chaining algorithm (FOL-BC-OR / FOL-BC-AND).
    Returns a generator yielding valid substitution dictionaries that satisfy the goals.
    """
    indent = "  " * depth
    if not goals:
        yield theta
        return

    # Pop the first goal from our current conjunct list
    first = goals[0]
    rest = goals[1:]

    # Apply existing substitutions to our target goal
    subst_first = substitute(theta, first)
    print(f"{indent}Checking sub-goal: {subst_first[0]}({', '.join(subst_first[1:])})")

    # Step A: Check if the sub-goal matches a direct ground fact in our KB
    for fact in facts:
        unified_theta = unify(subst_first, fact, theta)
        if unified_theta is not None:
            print(f"{indent} -> Matched Fact: {fact[0]}({', '.join(fact[1:])}) with bindings {unified_theta}")
            # Recursively solve remaining goals with the new bindings
            yield from fol_bc_ask(rules, facts, rest, unified_theta, depth + 1)

    # Step B: Check if any rule conclusion can derive this sub-goal
    for rule in rules:
        # Standardize variables dynamically to avoid name collisions across recursive stack calls
        standardized_rule = standardize_variables(rule, depth)

        unified_theta = unify(subst_first, standardized_rule.conclusion, theta)
        if unified_theta is not None:
            print(f"{indent} -> Rule Target Triggered: {standardized_rule.conclusion[0]} via rule premises")

            # Combine the rule's prerequisites with remaining open goals
            new_goals = list(standardized_rule.premises) + list(rest)
            yield from fol_bc_ask(rules, facts, new_goals, unified_theta, depth + 1)


def standardize_variables(rule, depth):
    """Applies a depth prefix to variables to keep rule instances unique in the recursive call stack"""
    def suffix(arg):
        return f"{arg}_{depth}" if is_variable(arg) else arg

    new_premises = [(p[0],) + tuple(suffix(a) for a in p[1:]) for p in rule.premises]
    new_conclusion = (rule.conclusion[0],) + tuple(suffix(a) for a in rule.conclusion[1:])
    return Clause(new_premises, new_conclusion)


def backward_chain(rules, facts, goal):
    """Wrapper function to invoke the recursive engine and format outputs cleanly"""
    print(f"Target goal to prove: {goal[0]}({', '.join(goal[1:])})\n")

    solutions = list(fol_bc_ask(rules, facts, [goal], {}))

    print("\n================ INFERENCE REPORT ================")
    if solutions:
        print(f"🎉 GOAL SUCCESS: Successfully proved {goal[0]}({', '.join(goal[1:])})!")
        print("Valid binding paths discovered:")
        for idx, sol in enumerate(solutions, 1):
            cleaned_sol = {k.split('_')[0]: v for k, v in sol.items() if not k.startswith('__')}
            print(f"  Path {idx}: {cleaned_sol}")
        return True
    else:
        print(f"❌ GOAL FAILURE: Could not prove {goal[0]}({', '.join(goal[1:])}) via backward reasoning.")
        return False

# ===========================================================================
# PARSING METHODS
# ===========================================================================

def parse_predicate(pred_str):
    """Parses 'Predicate(Arg1, Arg2)' into ('Predicate', 'Arg1', 'Arg2')"""
    match = re.match(r"(\w+)\s*\(([^)]+)\)", pred_str.strip())
    if not match:
        raise ValueError(f"Invalid predicate format: '{pred_str}'")
    pred_name = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (pred_name,) + tuple(args)

def parse_rule(rule_str):
    """Parses 'P1(x) & P2(y) => C(x)' into a Clause object"""
    if "=>" not in rule_str:
        raise ValueError(f"Invalid rule format (missing '=>' operator): '{rule_str}'")
    premises_part, conclusion_part = rule_str.split("=>")

    premises_list = [p.strip() for p in premises_part.split("&") if p.strip()]
    premises = [parse_predicate(p) for p in premises_list]
    conclusion = parse_predicate(conclusion_part.strip())

    return Clause(premises, conclusion)

# ===========================================================================
# MAIN INTERACTIVE ENTRY POINT
# ===========================================================================

print("=== INTERACTIVE FOL BACKWARD CHAINING ===\n")
print("Please provide the logic setup. You can press Enter to use the default values shown in brackets.\n")

# Input facts
def_facts = "American(Robert), Enemy(CountryA, America), Owns(CountryA, Missiles), Missile(Missiles)"
facts_input = input(f"Enter facts separated by commas\n[Default: {def_facts}]: ").strip()
if not facts_input:
    facts_input = def_facts

parsed_facts = []
for f in facts_input.split("),"):
    f_str = f.strip()
    if f_str:
        if not f_str.endswith(")"):
            f_str += ")"
        parsed_facts.append(parse_predicate(f_str))

# Input rules
def_rules = [
    "American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z) => Criminal(x)",
    "Enemy(z, America) => Hostile(z)",
    "Missile(x) => Weapon(x)",
    "Missile(y) & Owns(z, y) => Sells(Robert, y, z)"
]

parsed_rules = []
print("\nEnter rules one by one. Press Enter on an empty line when finished.")
rule_index = 1
while True:
    default_val = def_rules[rule_index - 1] if rule_index <= len(def_rules) else ""
    prompt_suffix = f" [Default: {default_val}]" if default_val else ""

    r_in = input(f"Rule {rule_index}{prompt_suffix}: ").strip()
    if not r_in:
        if default_val:
            r_in = default_val
        else:
            break
    try:
        parsed_rules.append(parse_rule(r_in))
        rule_index += 1
    except ValueError as e:
        print(f"Error: {e}. Please re-enter this rule using valid syntax or press enter on an empty line to finish.")

# Input Goal
def_goal = "Criminal(Robert)"
goal_input = input(f"\nEnter the goal to prove\n[Default: {def_goal}]: ").strip()
if not goal_input:
    goal_input = def_goal
parsed_goal = parse_predicate(goal_input)

# Execute Engine
print("\n================ RUNNING BACKWARD CHAINING ================")
backward_chain(parsed_rules, parsed_facts, parsed_goal)


=== INTERACTIVE FOL BACKWARD CHAINING ===

Please provide the logic setup. You can press Enter to use the default values shown in brackets.

Enter facts separated by commas
[Default: American(Robert), Enemy(CountryA, America), Owns(CountryA, Missiles), Missile(Missiles)]: 

Enter rules one by one. Press Enter on an empty line when finished.
Rule 1 [Default: American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z) => Criminal(x)]: 
Rule 2 [Default: Enemy(z, America) => Hostile(z)]: 
Rule 3 [Default: Missile(x) => Weapon(x)]: 
Rule 4 [Default: Missile(y) & Owns(z, y) => Sells(Robert, y, z)]: 
Rule 5: 

Enter the goal to prove
[Default: Criminal(Robert)]: 

================ RUNNING BACKWARD CHAINING ================
Target goal to prove: Criminal(Robert)

Checking sub-goal: Criminal(Robert)
 -> Rule Target Triggered: Criminal via rule premises
  Checking sub-goal: American(Robert)
   -> Matched Fact: American(Robert) with bindings {'x_0': 'Robert'}
    Checking sub-goal: Weapon(y_0)
     -> 

True